## Is model's semantic representation comparable to human's?
It is argued that models are not social, not emotional, and don't have physical body, wich are essential means in human language learning.

This project examines whether the associations between shape and Social, shape and Emotional Arousal, and shape and Body-Object Interaction (BOI) are weaker in model's representation of language, compared to human's representation of language.

## Data structure and variables
Socialness Ratings (1-7)
- Diveica, V., Pexman, P.M. & Binney, R.J. (2023). Quantifying social semantics: An inclusive definition of socialness and ratings for 8388 English words. Behav Res 55, 461–473. https://doi.org/10.3758/s13428-022-01810-x
- Ratings available here: https://github.com/DiveicaV/SocialnessNorms/blob/main/Data/Ratings_sum.stat.csv

Arousal and Size Ratings (1-9)
- Scott, G.G., Keitel, A., Becirspahic, M. et al. The Glasgow Norms: Ratings of 5,500 words on nine scales. Behav Res 51, 1258–1270 (2019). https://doi.org/10.3758/s13428-018-1099-3

BOI Ratings (1-7)
- Pexman, P.M., Muraki, E., Sidhu, D.M., Paul D. Siakaluk & Melvin J. Yap. (2019). Quantifying sensorimotor experience: Body–object interaction ratings for more than 9,000 English words. Behav Res 51, 453–466. https://doi.org/10.3758/s13428-018-1171-z



In [ ]:
#run this using GPU instead of CPU to reduce processing time
#on google colab, go to "runtime", go to "change runtime type", click on "GPU", hit "save"

In [ ]:
#install dependencies
!pip install transformers accelerate sentencepiece
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
from google.colab import files
import io
from scipy.stats import pearsonr
import sys
import time

## Load model

In [ ]:
#load the model + tokenizer
#model downloaded from hugginface: https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
model_name = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
if hasattr(model.config, 'num_hidden_layers'):
    num_layers = model.config.num_hidden_layers
elif hasattr(model.config, 'num_layers'):
    num_layers = model.config.num_layers
else:
    num_layers = "Could not determine the number of layers from model.config"

print(f"The language model has {num_layers} layers.")

The language model has 28 layers.


In [ ]:
#test the model out
text = ["This is a test sentence."]

inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}  #sends tensors to same device as model

with torch.no_grad():
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  #simple mean pool

embeddings.shape  #shape of the embedding vector that the model produced. 1024 dimensions

torch.Size([1, 1024])

In [ ]:
#see if notebook is operating on GPU
model.device  #GPU becasue it's "cuda"

device(type='cuda', index=0)

## Load datasets for words and find the overlapping words

In [ ]:
socialness_uploaded = files.upload()
socialness_df = pd.read_csv(io.BytesIO(socialness_uploaded['Socialness Ratings.csv']))

Saving Socialness Ratings.csv to Socialness Ratings.csv


In [ ]:
BOI_uploaded = files.upload()
BOI_df = pd.read_csv(io.BytesIO(BOI_uploaded['BOI Ratings.csv']))

Saving BOI Ratings.csv to BOI Ratings.csv


In [ ]:
Glasgow_uploaded = files.upload()
Glasgow_df = pd.read_csv(io.BytesIO(Glasgow_uploaded['Glasgow Ratings.csv']))

Saving Glasgow Ratings.csv to Glasgow Ratings.csv


In [ ]:
print("Socialness DataFrame Head:")
print(socialness_df.head())
print("\nBOI DataFrame Head:")
print(BOI_df.head())
print("\nArousalSize DataFrame Head:")
print(Glasgow_df.head())

Socialness DataFrame Head:
          Word      Mean        SD  Median  Min  Max   N
0      abandon  4.782609  1.857577     5.0    1    7  23
1  abandonment  4.681818  2.009178     5.0    1    7  22
2      abdomen  2.217391  1.999012     1.0    1    7  23
3    abdominal  2.285714  1.874643     1.0    1    7  21
4       abduct  5.000000  2.366432     6.0    1    7  21

BOI DataFrame Head:
        Word   N      Mean        SD
0     abacus  24  4.666667  2.098999
1    abdomen  23  5.434783  1.996044
2  abdominal  25  3.920000  2.271563
3     abduct  24  1.791667  1.693444
4      abide  25  1.760000  1.392839

ArousalSize DataFrame Head:
         Word  Length  AROU_M  AROU_SD  AROU_N  VAL_M  VAL_SD  VAL_N  DOM_M  \
0    abattoir       8   4.200    2.400      25  2.864   1.740     22  4.333   
1       abbey       5   3.125    2.342      32  5.781   1.268     32  4.667   
2  abbreviate      10   3.273    1.582      33  5.250   1.031     32  5.235   
3    abdicate       8   4.194    1.941     

In [ ]:
#finding overlapping words across the 3 datasets
socialness_words = set(socialness_df['Word'].str.lower())
BOI_words = set(BOI_df['Word'].str.lower())
arousalsize_words = set(Glasgow_df['Word'].str.lower())

common_words = socialness_words.intersection(arousalsize_words, BOI_words)

print(f"Number of common words across all datasets: {len(common_words)}")
print(f"First 10 common words: {list(common_words)[:10]}")

Number of common words across all datasets: 1529
First 10 common words: ['content', 'basket', 'greed', 'symbol', 'joke', 'exercise', 'fool', 'cite', 'psychologist', 'goggles']


In [ ]:
#merging data for overlapping words across three datasets

socialness_filtered_df = socialness_df[socialness_df['Word'].str.lower().isin(common_words)][['Word', 'Mean']]
socialness_filtered_df = socialness_filtered_df.rename(columns={'Mean': 'Socialness_Human'})

BOI_filtered_df = BOI_df[BOI_df['Word'].str.lower().isin(common_words)][['Word', 'Mean']]
BOI_filtered_df = BOI_filtered_df.rename(columns={'Mean': 'BOI_Human'})

arousalsize_filtered_df = Glasgow_df[Glasgow_df['Word'].str.lower().isin(common_words)][['Word', 'AROU_M', 'SIZE_M']]
arousalsize_filtered_df = arousalsize_filtered_df.rename(columns={'AROU_M': 'Arousal_Human'})
arousalsize_filtered_df = arousalsize_filtered_df.rename(columns={'SIZE_M': 'Size_Human'})


# Merge the filtered dataframes
merged_ratings_df = pd.merge(socialness_filtered_df, arousalsize_filtered_df, on='Word', how='inner')
merged_ratings_df = pd.merge(merged_ratings_df, BOI_filtered_df, on='Word', how='inner')

# Display the head of the merged DataFrame
print("Merged Ratings DataFrame Head:")
print(merged_ratings_df.head())

# Save the merged DataFrame to a CSV file
merged_ratings_df.to_csv('merged_ratings.csv', index=False)

Merged Ratings DataFrame Head:
        Word  Socialness_Human  Arousal_Human  Size_Human  BOI_Human
0    abdomen          2.217391          4.714       3.529   5.434783
1  abdominal          2.285714          3.875       3.788   3.920000
2     abduct          5.000000          4.677       4.438   1.791667
3      abide          4.904762          3.875       4.033   1.760000
4       able          3.458333          4.853       4.400   1.720000


## Semantic projection for the dimensions: Size, socialness, arousal, and BOI

In [ ]:
#define anchor words for shape dimension. these words are the ones that are commonly used sound symbolism research
size_low  = ["little", "small", "tiny"]
size_high = ["large", "big", "huge"]

#define anchor words for Socialness
socialness_low = ["machine", "unsociable", "solitude"]
socialness_high = ["people", "sociable", "together"]

#define anchor words for Arousal dimension. these words are from Warriner et al. (2013)
arousal_low  = [   "relaxed",    "calm", "sluggish",    "dull", "sleepy"]
arousal_high = ["stimulated", "excited", "frenzied", "jittery",  "awake", "aroused"]

#define anchor words for BOI
BOI_low  = ["immaterial", "untouchable",   "remote", "unreachable"]
BOI_high = [  "handheld", "manipulable", "tangible",   "reachable"]

In [ ]:
#define how to extract Qwen3 embedding vectors for each words
def embed_words_batch(words, batch_size=64):
    """
    Embed a list of words using Qwen3 in GPU batches.
    Returns dict: word -> normalized numpy embedding.
    """
    embeddings = {}
    model.eval()

    for i in range(0, len(words), batch_size):
        batch = words[i:i+batch_size]

        # Tokenize (CPU) then move to GPU
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            # (batch, hidden_dim)
            vecs = outputs.last_hidden_state.mean(dim=1)
            vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)

        # Move batch embeddings to CPU
        vecs = vecs.float().cpu().numpy()

        # Store into dictionary
        for w, v in zip(batch, vecs):
            embeddings[w] = v

        # Progress bar
        print(f"\rEmbedded {min(i+batch_size, len(words))}/{len(words)} words...", end='')

    print("\nBatch embedding complete.")
    return embeddings

In [ ]:
#define all words that we will get embedding vectors
all_words = merged_ratings_df['Word'].tolist()
anchor_words = size_low + size_high + socialness_low + socialness_high + arousal_low + arousal_high + BOI_low + BOI_high

#unique vocabulary (no repeats)
vocab = list(set(all_words + anchor_words))
print("Total unique words to embed:", len(vocab))

print(len(all_words))
print(len(anchor_words))

Total unique words to embed: 1532
1509
31


In [ ]:
#extracting embedding vectors for these words
emb_dict = embed_words_batch(vocab, batch_size=64)

Embedded 1532/1532 words...
Batch embedding complete.


In [ ]:
#functions to calculate shape and arousal score (dot product)
def normalize(vec):
    return vec / np.linalg.norm(vec)

def distance_with_concepts_multi_pairs_fast(word, low_anchors, high_anchors, emb_dict):
    v_word = emb_dict[word]
    #note that the individual word's vector (v_word) is normalized
    #this happens within the embed_words_batch function, where
    #torch.nn.functional.normalize(vecs, p=2, dim=1) is applied before the
    #embeddings are stored in emb_dict

    directions = []
    for h in high_anchors:
        for l in low_anchors:
            directions.append(emb_dict[h] - emb_dict[l])

    axis = normalize(np.mean(directions, axis=0))
    return np.dot(v_word, axis)

In [ ]:
#calculating size, socialness, arousal, and BOI scores using functions above
size_scores = []
socialness_scores = []
arousal_scores = []
BOI_scores = []

total = len(merged_ratings_df)

for index, row in merged_ratings_df.iterrows():
    word = row['Word']

    progress_str = f'\rprocessing {index+1}/{total} words...'
    sys.stdout.write(progress_str)
    sys.stdout.flush()

    if word in emb_dict:
        sizescore = distance_with_concepts_multi_pairs_fast(
            word,
            size_low,
            size_high,
            emb_dict)
    else:
        sizescore = "na"

    if word in emb_dict:
        socialnessscore = distance_with_concepts_multi_pairs_fast(
            word,
            socialness_low,
            socialness_high,
            emb_dict)
    else:
        socialnessscore = "na"

    if word in emb_dict:
        arousalscore = distance_with_concepts_multi_pairs_fast(
            word,
            arousal_low,
            arousal_high,
            emb_dict)
    else:
        arousalscore = "na"

    if word in emb_dict:
        BOIscore = distance_with_concepts_multi_pairs_fast(
            word,
            BOI_low,
            BOI_high,
            emb_dict)
    else:
       BOIscore = "na"

    size_scores.append(sizescore)
    socialness_scores.append(socialnessscore)
    arousal_scores.append(arousalscore)
    BOI_scores.append(BOIscore)

print()

Qwen3_scores_df = pd.DataFrame({
    'Word': merged_ratings_df['Word'],
    'Size_Model': size_scores,
    'Socialness_Model': socialness_scores,
    'Arousal_Model': arousal_scores,
    'BOI_Model': BOI_scores,
})

processing 1509/1509 words...


In [ ]:
Qwen3_scores_df.head()

,Word,Size_Model,Socialness_Model,Arousal_Model,BOI_Model
0,abdomen,0.056148,-0.020447,-0.030348,-0.064852
1,abdominal,0.059696,-0.029871,-0.023159,-0.054357
2,abduct,0.028452,0.006371,0.013908,-0.067523
3,abide,-0.006609,-0.051504,-0.027777,-0.088694
4,able,0.051648,0.039477,0.048050,-0.062307


In [ ]:
#append Qwen3_scores_df and merged_ratings_df
merged_ratings_scores_df = pd.merge(merged_ratings_df, Qwen3_scores_df, on='Word')
print(merged_ratings_scores_df.head())
print(len(merged_ratings_scores_df))
merged_ratings_scores_df.to_csv("merged_ratings_scores_df.csv", index=False)

        Word  Socialness_Human  Arousal_Human  Size_Human  BOI_Human  \
0    abdomen          2.217391          4.714       3.529   5.434783   
1  abdominal          2.285714          3.875       3.788   3.920000   
2     abduct          5.000000          4.677       4.438   1.791667   
3      abide          4.904762          3.875       4.033   1.760000   
4       able          3.458333          4.853       4.400   1.720000   

   Size_Model  Socialness_Model  Arousal_Model  BOI_Model  
0    0.056148         -0.020447      -0.030348  -0.064852  
1    0.059696         -0.029871      -0.023159  -0.054357  
2    0.028452          0.006371       0.013908  -0.067523  
3   -0.006609         -0.051504      -0.027777  -0.088694  
4    0.051648          0.039477       0.048050  -0.062307  
1509


## Correlstions between all variables

In [ ]:
from scipy.stats import spearmanr

# Columns to include in the combined matrix
cols = ['Socialness_Human', 'Arousal_Human', 'Size_Human', 'BOI_Human', 'Size_Model', 'Socialness_Model', 'Arousal_Model', 'BOI_Model']

# Convert columns to numeric (coerce errors to NaN)
for col in cols:
    merged_ratings_scores_df[col] = pd.to_numeric(merged_ratings_scores_df[col], errors='coerce')

# Drop rows with ANY missing values in these columns
df_corr = merged_ratings_scores_df.dropna(subset=cols)

# Create empty result DataFrame
combined_matrix = pd.DataFrame(index=cols, columns=cols, dtype=object)

# Compute Spearman correlations for all pairs
for i in cols:
    for j in cols:
        if pd.api.types.is_numeric_dtype(df_corr[i]) and pd.api.types.is_numeric_dtype(df_corr[j]):
            corr, pval = spearmanr(df_corr[i], df_corr[j])
            combined_matrix.loc[i, j] = f"{corr:.3f} ({pval:.3f})"
        else:
            combined_matrix.loc[i, j] = "N/A"

print("Combined Spearman Correlation Coefficient (p-value) Matrix:")
print(combined_matrix)

Combined Spearman Correlation Coefficient (p-value) Matrix:
                 Socialness_Human   Arousal_Human      Size_Human  \
Socialness_Human    1.000 (0.000)   0.300 (0.000)   0.475 (0.000)   
Arousal_Human       0.300 (0.000)   1.000 (0.000)   0.503 (0.000)   
Size_Human          0.475 (0.000)   0.503 (0.000)   1.000 (0.000)   
BOI_Human          -0.219 (0.000)  -0.149 (0.000)  -0.326 (0.000)   
Size_Model          0.128 (0.000)   0.286 (0.000)   0.441 (0.000)   
Socialness_Model    0.317 (0.000)   0.113 (0.000)   0.143 (0.000)   
Arousal_Model       0.243 (0.000)   0.303 (0.000)   0.258 (0.000)   
BOI_Model          -0.080 (0.002)   0.014 (0.581)  -0.180 (0.000)   

                       BOI_Human      Size_Model Socialness_Model  \
Socialness_Human  -0.219 (0.000)   0.128 (0.000)    0.317 (0.000)   
Arousal_Human     -0.149 (0.000)   0.286 (0.000)    0.113 (0.000)   
Size_Human        -0.326 (0.000)   0.441 (0.000)    0.143 (0.000)   
BOI_Human          1.000 (0.000)  -0.114 (

## Analysis
Model 1: Size ~ Socialness * Group

Model 2: Size ~ Arousal * Group

Model 3: Size ~ BOI * Group

Hypothesis: the associations between size and socialness, arousal, and BOI will be stronger for human than for model

In [ ]:
#data processing for analyses

#turn into numeric
merged_ratings_scores_df['Socialness_Human'] = pd.to_numeric(merged_ratings_scores_df['Socialness_Human'], errors='coerce')
merged_ratings_scores_df['Arousal_Human'] = pd.to_numeric(merged_ratings_scores_df['Arousal_Human'], errors='coerce')
merged_ratings_scores_df['BOI_Human'] = pd.to_numeric(merged_ratings_scores_df['BOI_Human'], errors='coerce')
merged_ratings_scores_df['Size_Human'] = pd.to_numeric(merged_ratings_scores_df['Size_Human'], errors='coerce')

merged_ratings_scores_df['Socialness_Model'] = pd.to_numeric(merged_ratings_scores_df['Socialness_Model'], errors='coerce')
merged_ratings_scores_df['Arousal_Model'] = pd.to_numeric(merged_ratings_scores_df['Arousal_Model'], errors='coerce')
merged_ratings_scores_df['BOI_Model'] = pd.to_numeric(merged_ratings_scores_df['BOI_Model'], errors='coerce')
merged_ratings_scores_df['Size_Model'] = pd.to_numeric(merged_ratings_scores_df["Size_Model"], errors="coerce")

#scale (for easier interpretation)
from sklearn.preprocessing import StandardScaler
merged_ratings_scores_df[
    ["SocialnessScaled_Human", "ArousalScaled_Human", "BOIScaled_Human", "SizeScaled_Human",
     "SocialnessScaled_Model", "ArousalScaled_Model", "BOIScaled_Model", "SizeScaled_Model"]
    ] = StandardScaler().fit_transform(merged_ratings_scores_df[
        ["Socialness_Human", "Arousal_Human", "BOI_Human", "Size_Human",
         "Socialness_Model", "Arousal_Model", "BOI_Model", "Size_Model"]])

In [ ]:
#convert to long form with additional column: Group (Human vs. Model)

value_vars = [c for c in merged_ratings_scores_df.columns if "_" in c and c not in ["Word"]]

df_long = merged_ratings_scores_df.melt(id_vars=["Word"], value_vars=value_vars, var_name="variable", value_name="value")

# Step 3: Split variable into measurement and group
df_long[["measurement", "group"]] = df_long["variable"].str.rsplit("_", n=1, expand=True)

# Step 4: Pivot to have one column per measurement
df_final = df_long.pivot_table(index=["Word", "group"], columns="measurement", values="value").reset_index()


In [ ]:
print(df_final)

measurement       Word  group   Arousal  ArousalScaled       BOI  BOIScaled  \
0              abdomen  Human  4.714000       0.022675  5.434783   1.510893   
1              abdomen  Model -0.030348      -1.299337 -0.064852   0.068758   
2            abdominal  Human  3.875000      -0.772427  3.920000   0.466591   
3            abdominal  Model -0.023159      -1.143171 -0.054357   0.330494   
4               abduct  Human  4.677000      -0.012390  1.791667  -1.000696   
...                ...    ...       ...            ...       ...        ...   
3013             young  Model  0.034911       0.118300 -0.072941  -0.132977   
3014             youth  Human  5.548000       0.813038  3.296296   0.036606   
3015             youth  Model  0.079145       1.079209 -0.067076   0.013275   
3016             zebra  Human  4.818000       0.121233  4.000000   0.521744   
3017             zebra  Model  0.050325       0.453137 -0.095536  -0.696510   

measurement      Size  SizeScaled  Socialness  Soci

In [ ]:
#Model 1
import statsmodels.formula.api as smf

model1 = smf.ols("SizeScaled ~ SocialnessScaled * group", data=df_final).fit()

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:             SizeScaled   R-squared:                       0.115
Model:                            OLS   Adj. R-squared:                  0.114
Method:                 Least Squares   F-statistic:                     131.0
Date:                Sun, 22 Feb 2026   Prob (F-statistic):           9.33e-80
Time:                        21:56:48   Log-Likelihood:                -4097.4
No. Observations:                3018   AIC:                             8203.
Df Residuals:                    3014   BIC:                             8227.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

In [ ]:
#Model 2

model2 = smf.ols("SizeScaled ~ ArousalScaled * group", data=df_final).fit()

print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:             SizeScaled   R-squared:                       0.172
Model:                            OLS   Adj. R-squared:                  0.171
Method:                 Least Squares   F-statistic:                     208.8
Date:                Sun, 22 Feb 2026   Prob (F-statistic):          4.61e-123
Time:                        21:59:59   Log-Likelihood:                -3997.4
No. Observations:                3018   AIC:                             8003.
Df Residuals:                    3014   BIC:                             8027.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [ ]:
#Model 3

model3 = smf.ols("SizeScaled ~ BOIScaled * group", data=df_final).fit()

print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:             SizeScaled   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     78.19
Date:                Sun, 22 Feb 2026   Prob (F-statistic):           1.06e-48
Time:                        22:00:34   Log-Likelihood:                -4169.3
No. Observations:                3018   AIC:                             8347.
Df Residuals:                    3014   BIC:                             8371.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               